# 01 — In-context learning (ICL)

Classifying word senses into supersenses by **prompting** `Qwen2.5-7B-Instruct` (4-bit), without training.
All choices are tuned on **`rand-dev`**, changing **one factor at a time** (see `docs/prompt_literature.md`):

1. prompt language: English instruction and labels vs everything in French;
2. number of demonstrations: 0 / 5 / 25 / 50;
3. demonstration selection: random vs balanced over classes vs most similar definitions;
4. with vs without the lemma.

The best configuration is run once on `rand-test` and `freq-test` at the end of the notebook.
Runtime: GPU (T4).

In [ ]:
# Setup: in Colab, clone (or update) the repo so that `src/` is importable; locally, run from the repo root.
import os, sys
REPO_URL = "https://github.com/bkadyrbay-hash/lab1-supersense-llm.git"
BRANCH = "a/icl-experiments"  # set to your own branch to test code that is not merged yet
if "google.colab" in sys.modules:
    REPO_DIR = "/content/lab1-supersense-llm"
    if not os.path.exists(REPO_DIR):
        !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
!pip install -q bitsandbytes  # 4-bit weights; the rest is preinstalled in Colab

In [ ]:
# All hyperparameters of this notebook.
CONFIG = {
    "model": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",  # see docs/model_choice.md
    "tuning_split": "rand-dev",
    "max_new_tokens": 10,  # the longest label is 5 tokens; a longer answer is cut and parsed anyway
    "seed": 42,
}

In [ ]:
import copy
import time

import pandas as pd
import torch
from tqdm.auto import tqdm
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.data import HYPERSENSES, SUPER2HYPER, SUPERSENSES, get_split, load_data
from src.evaluate import RESULTS_DIR, evaluate, log_experiment, paired_bootstrap, save_predictions
from src.prompts import ShotSelector, build_prompt, parse_answer

# Hides a warning printed at every generate() call (the model config sets max_length, we set max_new_tokens).
transformers.logging.set_verbosity_error()
torch.manual_seed(CONFIG["seed"])
df = load_data()
train = get_split(df, "train")
dev = get_split(df, CONFIG["tuning_split"])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["model"], device_map="auto", dtype=torch.float16)
model.eval()
print(f"GPU memory used by the weights: {torch.cuda.memory_allocated() / 1e9:.1f} GB" if torch.cuda.is_available() else "CPU")

## Prediction by generation

The model answers freely (greedy decoding) and `parse_answer` maps the text to a label.
With `random` or `by_class` demonstrations, the instruction and the demonstrations are **the same for every query**:
we encode this prefix once and reuse its key/value cache, so each query only costs its own few tokens
(about 10× faster with 50 demonstrations — less GPU time and energy).

In [ ]:
def encode(messages, generation_prompt):
    """Token ids of a chat, as the model sees it."""
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=generation_prompt, tokenize=False)
    return tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)


@torch.no_grad()
def generate_answers(eval_df, selector, lang, with_lemma):
    """Raw generated answer for each row of `eval_df`."""
    prefix_cache = None
    answers = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        messages = build_prompt(row, selector(row), lang, with_lemma)
        input_ids = encode(messages, generation_prompt=True)
        cache = None
        if selector.strategy != "similar":
            if prefix_cache is None:  # first query: encode the shared prefix (instruction + demonstrations)
                prefix_ids = encode(messages[:-1], generation_prompt=False)
                assert torch.equal(input_ids[0, : prefix_ids.shape[1]], prefix_ids[0])
                prefix_cache = model(prefix_ids, use_cache=True).past_key_values
            cache = copy.deepcopy(prefix_cache)  # generate() extends the cache in place
        output = model.generate(input_ids, attention_mask=torch.ones_like(input_ids), past_key_values=cache,
                                max_new_tokens=CONFIG["max_new_tokens"], do_sample=False,
                                temperature=None, top_p=None, top_k=None, pad_token_id=tokenizer.eos_token_id)
        answers.append(tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True))
    return answers

In [ ]:
def config_id(lang, n_shots, shot_selection, with_lemma):
    """Readable run name, e.g. en_25shot_by_class_lemma."""
    shots = f"{n_shots}shot" + (f"_{shot_selection}" if n_shots else "")
    return f"{lang}_{shots}_{'lemma' if with_lemma else 'nolemma'}"


def run(lang, n_shots, shot_selection="by_class", with_lemma=True, split=CONFIG["tuning_split"]):
    """Predict, score, save and log one configuration; reuse the saved predictions if it was already run."""
    cid = config_id(lang, n_shots, shot_selection, with_lemma)
    path = RESULTS_DIR / "predictions" / f"icl__{cid}__{split}.csv"
    fresh = not path.exists()
    if not fresh:  # Colab sessions are cut regularly: never pay twice for the same run
        preds = pd.read_csv(path, keep_default_na=False).replace("", None)
    else:
        eval_df = get_split(df, split)
        start = time.time()
        answers = generate_answers(eval_df, ShotSelector(train, n_shots, shot_selection, CONFIG["seed"]), lang, with_lemma)
        runtime = (time.time() - start) / 60
        preds = eval_df[["sense_id", "lemma", "definition"]].assign(
            gold_supersense=eval_df["supersense"], pred_supersense=[parse_answer(a, lang) for a in answers],
            gold_hypersense=eval_df["hypersense"], raw_output=answers)
        preds.insert(6, "pred_hypersense", preds["pred_supersense"].map(SUPER2HYPER))
        save_predictions(preds, "icl", cid, split)
    sup = evaluate(preds["gold_supersense"].tolist(), preds["pred_supersense"].tolist(), SUPERSENSES)
    hyp = evaluate(preds["gold_hypersense"].tolist(), preds["pred_hypersense"].tolist(), HYPERSENSES)
    if fresh:
        log_experiment({"author": "A", "method": "icl", "config_id": cid, "split": split, "model": CONFIG["model"],
                        "with_lemma": with_lemma, "prompt_lang": lang, "n_shots": n_shots,
                        "shot_selection": shot_selection if n_shots else "", "decoding": "generation",
                        "accuracy_super": round(sup["accuracy"], 4), "macro_f1_super": round(sup["macro_f1"], 4),
                        "accuracy_hyper": round(hyp["accuracy"], 4), "macro_f1_hyper": round(hyp["macro_f1"], 4),
                        "invalid_rate": round(sup["invalid_rate"], 4), "n": sup["n"], "runtime_min": round(runtime, 1)})
    print(f"{cid:35s} macro-F1 {sup['macro_f1']:.3f}  acc {sup['accuracy']:.3f}  "
          f"hyper macro-F1 {hyp['macro_f1']:.3f}  invalid {sup['invalid_rate']:.1%}")
    return {"config_id": cid, "macro_f1": sup["macro_f1"], "accuracy": sup["accuracy"],
            "hyper_macro_f1": hyp["macro_f1"], "invalid_rate": sup["invalid_rate"], "preds": preds}


def compare(a, b):
    """Print how often b beats a on bootstrap resamples of the evaluation set."""
    share = paired_bootstrap(a["preds"]["gold_supersense"], a["preds"]["pred_supersense"],
                             b["preds"]["pred_supersense"], SUPERSENSES)
    print(f"{b['config_id']} beats {a['config_id']} in {share:.0%} of bootstrap resamples")

## 1. Prompt language

English instruction and English labels vs French instruction and French labels (inputs are French in both cases).
Compared without demonstrations (the instruction is all the model sees) and with 5 balanced demonstrations.

In [ ]:
lang_runs = {(lang, n): run(lang, n) for lang in ["en", "fr"] for n in [0, 5]}
compare(lang_runs["en", 0], lang_runs["fr", 0])
compare(lang_runs["en", 5], lang_runs["fr", 5])
LANG = max(["en", "fr"], key=lambda l: lang_runs[l, 0]["macro_f1"] + lang_runs[l, 5]["macro_f1"])
print("Prompt language kept:", LANG)

## 2. Number of demonstrations

Balanced demonstrations (`by_class`): 25 shots = one per class, 50 = two per class.

In [ ]:
shot_runs = {n: run(LANG, n) for n in [0, 5, 25, 50]}
N_SHOTS = max(shot_runs, key=lambda n: shot_runs[n]["macro_f1"])
compare(shot_runs[0], shot_runs[N_SHOTS])
print("Number of shots kept:", N_SHOTS)

## 3. Choice of the demonstrations

`random`: a random sample of train (follows the train label distribution);
`by_class`: balanced over labels; `similar`: for each query, the training definitions most similar to its definition
(TF-IDF on character n-grams).

In [ ]:
selection_runs = {s: run(LANG, N_SHOTS, s) for s in ["random", "by_class", "similar"]}
SELECTION = max(selection_runs, key=lambda s: selection_runs[s]["macro_f1"])
for s in selection_runs:
    if s != SELECTION:
        compare(selection_runs[s], selection_runs[SELECTION])
print("Selection kept:", SELECTION)

## 4. Is the lemma useful?

In [ ]:
lemma_runs = {w: run(LANG, N_SHOTS, SELECTION, with_lemma=w) for w in [True, False]}
compare(lemma_runs[False], lemma_runs[True])
WITH_LEMMA = lemma_runs[True]["macro_f1"] >= lemma_runs[False]["macro_f1"]

## Summary of the tuning runs

In [ ]:
metrics = pd.read_csv(RESULTS_DIR / "metrics.csv")
metrics[(metrics["method"] == "icl") & (metrics["split"] == CONFIG["tuning_split"])].drop_duplicates(
    "config_id", keep="last").sort_values("macro_f1_super", ascending=False)[
    ["config_id", "macro_f1_super", "accuracy_super", "macro_f1_hyper", "invalid_rate", "runtime_min"]]

## Conclusions on rand-dev

- (to be written after the runs)

## Final evaluation on the test sets

Run **once**, with the configuration chosen above (A8).

In [ ]:
# for split in ["rand-test", "freq-test"]:
#     run(LANG, N_SHOTS, SELECTION, WITH_LEMMA, split=split)

## Download the results

Prediction files and log rows are zipped and downloaded, then committed to the branch from a computer.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files
    !cd /content/lab1-supersense-llm && zip -qr /content/icl_results.zip results
    files.download("/content/icl_results.zip")